# 📧 SMS Spam Classifier — Fine-Tuning DistilBERT
**Course Term Project** | Fine-tuning a pre-trained transformer model for binary text classification

**Model:** `distilbert-base-uncased`  
**Task:** Spam vs. Ham (Not Spam) classification  
**Dataset:** SMS Spam Collection (UCI / HuggingFace)

---
### Runtime Setup
Go to **Runtime → Change runtime type → T4 GPU** before running.

## Step 1: Install Dependencies

In [ ]:
!pip install transformers datasets scikit-learn -q

## Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Step 3: Load the Dataset

In [ ]:
# Load the SMS Spam Collection dataset from HuggingFace
dataset = load_dataset('sms_spam')
print(dataset)
print('\nSample entry:')
print(dataset['train'][0])

In [ ]:
# Explore label distribution
df = pd.DataFrame(dataset['train'])
print('Label distribution:')
print(df['label'].value_counts())
print('\n0 = Ham (not spam), 1 = Spam')

# Preview a few examples
print('\n--- Example HAM messages ---')
for msg in df[df['label']==0]['sms'].head(3):
    print(f'  {msg[:80]}...')

print('\n--- Example SPAM messages ---')
for msg in df[df['label']==1]['sms'].head(3):
    print(f'  {msg[:80]}...')

## Step 4: Split into Train / Validation / Test Sets

In [ ]:
# The dataset only has a 'train' split — we'll split it ourselves
split1 = dataset['train'].train_test_split(test_size=0.2, seed=42)
split2 = split1['test'].train_test_split(test_size=0.5, seed=42)

train_dataset = split1['train']      # 80%
val_dataset   = split2['train']      # 10%
test_dataset  = split2['test']       # 10%

print(f'Train size:      {len(train_dataset)}')
print(f'Validation size: {len(val_dataset)}')
print(f'Test size:       {len(test_dataset)}')

## Step 5: Tokenize the Text

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['sms'], truncation=True, max_length=128)

# Apply tokenization
train_tok = train_dataset.map(tokenize, batched=True)
val_tok   = val_dataset.map(tokenize, batched=True)
test_tok  = test_dataset.map(tokenize, batched=True)

print('Tokenization complete.')
print('Columns:', train_tok.column_names)

## Step 6: Load Pre-trained Model & Modify Output Layer

In [ ]:
# Load DistilBERT with a classification head (2 output classes: ham/spam)
# AutoModelForSequenceClassification automatically replaces the output
# layer with a new linear layer matching our num_labels
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'HAM', 1: 'SPAM'},
    label2id={'HAM': 0, 'SPAM': 1}
)

print('Model loaded!')
print(f'Output layer: {model.classifier}')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## Step 7: Define Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='binary')
    return {'accuracy': acc, 'f1': f1}

## Step 8: Configure Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='./logs',
    logging_steps=50,
    report_to='none'
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Training config ready.')

## Step 9: Train the Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print('Starting training...')
trainer.train()

## Step 10: Evaluate on Test Set

In [ ]:
results = trainer.evaluate(test_tok)
print('\n=== Test Set Results ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Detailed classification report
preds_output = trainer.predict(test_tok)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print('\n=== Classification Report ===')
print(classification_report(labels, preds, target_names=['Ham', 'Spam']))

## Step 11: Test with Custom Inputs

In [ ]:
def predict_message(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1).item()
    label = 'SPAM 🚨' if pred == 1 else 'HAM ✅'
    probs = torch.softmax(logits, dim=-1)[0]
    print(f'Message: "{text[:70]}"')
    print(f'Prediction: {label} (confidence: {probs[pred]:.2%})\n')

# Try some examples
predict_message('Congratulations! You won a FREE iPhone. Click here to claim NOW!')
predict_message('Hey, are you coming to class tomorrow?')
predict_message('URGENT: Your bank account has been compromised. Call 1-800-555-0199')
predict_message('Can you pick up some groceries on your way home?')

## Step 12: Save the Model
*(Optional: save to Google Drive for later use)*

In [ ]:
# Save locally in Colab
model.save_pretrained('./spam_classifier_model')
tokenizer.save_pretrained('./spam_classifier_model')
print('Model saved to ./spam_classifier_model')

# Optional: mount Google Drive and copy
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./spam_classifier_model /content/drive/MyDrive/

---
## Summary
| Component | Choice |
|-----------|--------|
| Base model | `distilbert-base-uncased` (66M params) |
| Dataset | SMS Spam Collection (~5,500 messages) |
| Fine-tuning approach | Full fine-tuning (all layers) |
| Output layer | Linear(768 → 2) classification head |
| Optimizer | AdamW with weight decay |
| Epochs | 3 |
| Expected accuracy | ~98–99% |
| Expected F1 (spam) | ~95–97% |